In [1]:
!pip install transformers
!pip install datasets
!pip install torch
!pip install scikit-learn
!pip install matplotlib
!pip install seaborn

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from datasets import load_dataset
from transformers import BertTokenizer, BertForSequenceClassification
from transformers import Trainer, TrainingArguments

from sklearn.metrics import accuracy_score, precision_recall_fscore_support

In [72]:
from datasets import load_dataset
dataset = load_dataset("mediabiasgroup/mbib-base")

In [4]:
dataset

DatasetDict({
    cognitive_bias: Dataset({
        features: ['id', 'text', 'label', 'dataset_id'],
        num_rows: 7092
    })
    fake_news: Dataset({
        features: ['id', 'text', 'label', 'dataset_id'],
        num_rows: 8542
    })
    gender_bias: Dataset({
        features: ['id', 'text', 'label', 'dataset_id'],
        num_rows: 17940
    })
    hate_speech: Dataset({
        features: ['id', 'text', 'label', 'dataset_id'],
        num_rows: 339010
    })
    linguistic_bias: Dataset({
        features: ['id', 'text', 'label', 'dataset_id'],
        num_rows: 401862
    })
    political_bias: Dataset({
        features: ['id', 'text', 'label', 'dataset_id'],
        num_rows: 17704
    })
    racial_bias: Dataset({
        features: ['id', 'text', 'label', 'dataset_id'],
        num_rows: 9788
    })
    text_level_bias: Dataset({
        features: ['id', 'text', 'label', 'dataset_id'],
        num_rows: 9018
    })
})

In [73]:
gender = dataset["gender_bias"]
racial = dataset["racial_bias"]
political = dataset["political_bias"]
text_level=dataset["text_level_bias"]

In [74]:
gender[0]

{'id': '107-9295',
 'text': 'MENTION4637 I am so sorry that you re in this situation ',
 'label': 0,
 'dataset_id': 107}

dataset object from Hugging Face Dataset

Actually we don’t always need to convert to DataFrame.

Sometimes we directly use the dataset object.

But for preprocessing and exploration, DataFrames are much easier.

In [75]:
import pandas as pd

gender_df = pd.DataFrame(gender)
racial_df = pd.DataFrame(racial)
political_df = pd.DataFrame(political)
text_level_df = pd.DataFrame(text_level)

In [76]:
#head() shows the top rows of a dataset.
gender_df.head(10)

,id,text,label,dataset_id
0,107-9295,MENTION4637 I am so sorry that you re in this ...,0,107
1,107-10267,Being a Boomer is Terribly Difficult Task Sinc...,0,107
2,107-11607,MENTION3217 Big accomplishments west point and...,0,107
3,107-1052,I ve never met a kid who can write poetry,0,107
4,107-10961,Talk the talk but can t walk the walk MKR,0,107
5,106-205,Ed Zimmerman made a pledge to not attend all m...,0,106
6,107-11125,she parties like a rockstar but loves like a ...,0,107
7,107-5065,and Mary A is the fucking inventor of automob...,0,107
8,107-6679,MENTION513 NPO can take a year,0,107
9,075-1276,stop raising your voice citizen shoots and is ...,0,75


In [9]:
text_level_df.head()

,id,text,label,dataset_id
0,076-27461,Conservatives and fascists kill people Antifa ...,1,76
1,076-14487,I don t see how you can discount the accuracy ...,0,76
2,076-10304,Snapshots 1 This Post archive org i redd ...,0,76
3,076-22865,critiques of leftist ideology Do you really...,1,76
4,076-22208,I can t deny that those same antifa faggots th...,1,76


In [10]:
#show last 10 rows
gender_df.tail()

,id,text,label,dataset_id
17935,075-2079,liberal logic not all muslims are bad not all ...,1,75
17936,075-2406,no jews are not and never will be on our side ...,1,75
17937,075-3342,the religion called judaism is satanic anyone ...,1,75
17938,075-1474,im sorry but a random nurse isnt privy to the ...,1,75
17939,107-661,I do not believe a woman can do everything a m...,1,107


In [11]:
gender_df.shape

(17940, 4)

In [12]:
gender_df.columns

Index(['id', 'text', 'label', 'dataset_id'], dtype='object')

In [13]:
#check Label Distribution
text_level_df['label'].value_counts()

,count
label,
1,4509
0,4509


In [23]:
#check Label Distribution
gender_df['label'].value_counts()

,count
label,
0,8970
1,8970


In [77]:
gender_df["bias_type"] = "gender"
racial_df["bias_type"] = "racial"
political_df["bias_type"] = "political"
text_level_df["bias_type"] = "text_level"

In [78]:
gender_pos = gender_df[gender_df["label"] == 1].copy()
gender_pos["bias_type"] = "gender"

racial_pos = racial_df[racial_df["label"] == 1].copy()
racial_pos["bias_type"] = "racial"

political_pos = political_df[political_df["label"] == 1].copy()
political_pos["bias_type"] = "political"

no_bias_pos = text_level_df[text_level_df["label"] == 0].copy()
no_bias_pos["bias_type"] = "no_bias"

In [79]:
combined_df = pd.concat([
    gender_pos[["text", "bias_type"]],
    racial_pos[["text", "bias_type"]],
    political_pos[["text", "bias_type"]],
    no_bias_pos[["text", "bias_type"]]
], ignore_index=True)

print(combined_df["bias_type"].value_counts())
display(combined_df.head())

bias_type
gender       8970
political    8852
racial       4894
no_bias      4509
Name: count, dtype: int64


,text,bias_type
0,Most men come to a sexual situation in a state...,gender
1,wow gay people truly are mentally ill after al...,gender
2,Why do female managers think they are god s gr...,gender
3,yea my wife is a nurse and she audibly sighed ...,gender
4,MENTION772 on Radio4 now I think like a man wi...,gender


In [80]:
print(combined_df.isnull().sum())

text         0
bias_type    0
dtype: int64


In [81]:
label_map = {
    "gender": 1,
    "racial": 2,
    "political": 3,
    "no_bias": 0
}

combined_df["label"] = combined_df["bias_type"].map(label_map)
print(combined_df.head())

                                                text bias_type  label
0  Most men come to a sexual situation in a state...    gender      1
1  wow gay people truly are mentally ill after al...    gender      1
2  Why do female managers think they are god s gr...    gender      1
3  yea my wife is a nurse and she audibly sighed ...    gender      1
4  MENTION772 on Radio4 now I think like a man wi...    gender      1


In [82]:
print(combined_df[["bias_type", "label"]].drop_duplicates())
print(combined_df["label"].value_counts())

       bias_type  label
0         gender      1
8970      racial      2
13864  political      3
22716    no_bias      0
label
1    8970
3    8852
2    4894
0    4509
Name: count, dtype: int64


In [83]:
combined_df.tail(10)

,text,bias_type,label
27215,Shit glad we can finally cheat it s only the 2...,no_bias,0
27216,I saw the movie last Friday around 4 30 and th...,no_bias,0
27217,Hey OxidadoGuillermez Thank you for your submi...,no_bias,0
27218,Or at the very least support their local newsp...,no_bias,0
27219,Your submission or comment was removed for the...,no_bias,0
27220,I love when drama gets posted on and is immedi...,no_bias,0
27221,that is a joke you post didnt actually get rem...,no_bias,0
27222,I dont understand what you re trying to say Ma...,no_bias,0
27223,Only people like Viktor Medvedchuk who is cons...,no_bias,0
27224,Wow Fuck all those people And then Reddit will...,no_bias,0


In [84]:
combined_df = combined_df[["text", "label"]]

In [85]:
combined_df.shape

(27225, 2)

In [86]:
combined_df["label"].value_counts()

,count
label,
1,8970
3,8852
2,4894
0,4509


In [31]:
combined_df = combined_df.sample(frac=1, random_state=42).reset_index(drop=True)

In [87]:
combined_df.head(10)

,text,label
0,Most men come to a sexual situation in a state...,1
1,wow gay people truly are mentally ill after al...,1
2,Why do female managers think they are god s gr...,1
3,yea my wife is a nurse and she audibly sighed ...,1
4,MENTION772 on Radio4 now I think like a man wi...,1
5,MENTION3489 i dont expect that from a woman I ...,1
6,RT MENTION2039 I m not sexist but I want a gir...,1
7,war guy i guess ur right but ypu gotta admit j...,1
8,my mom is a clerk and she deals with this stuf...,1
9,throw guns in a lock box if friends are coming...,1


In [88]:
from sklearn.model_selection import train_test_split

train_texts, test_texts, train_labels, test_labels = train_test_split(
    combined_df["text"].tolist(),
    combined_df["label"].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=combined_df["label"]
)

In [89]:
print("Train texts:", len(train_texts))
print("Test texts:", len(test_texts))

Train texts: 21780
Test texts: 5445


In [90]:
train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_texts,
    train_labels,
    test_size=0.1,
    random_state=42,
    stratify=train_labels
)

In [91]:
print("Train texts:", len(train_texts))
print("Test texts:", len(test_texts))
print("Test texts:", len(val_texts))

Train texts: 19602
Test texts: 5445
Test texts: 2178


it randomly selects rows, but keeps their original index numbers.

In [92]:
val_texts[:5]

['The clinging vine wife is justified provided she clings sweetly enough to please her husband',
 'I m also wrong because I think a massive tax scheme to shape government influence will do net zero to impact it   The climate is changing Man may well be a major factor   Cap and trade has nothing to do with stopping it and everything to do with controlling economies ',
 'Got me I m an eminent educator at Happy Giraffe International Language School ',
 'im not saying all jews are bad but the jews in power are greedy long nose tribe rats period',
 'my wife is a labor and delivery nurse she would go crazy to try this on my vive how do i get my hands on this ']

Tokenization converts text into numbers that models like RoBERT can understand.

In [93]:
!pip install transformers datasets evaluate scikit-learn -q

In [94]:
from transformers import RobertaTokenizer

In [95]:
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

In [31]:
train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=128)
val_encodings = tokenizer(val_texts, truncation=True, padding=True, max_length=128)
test_encodings = tokenizer(test_texts, truncation=True, padding=True, max_length=128)

In [96]:
print(len(train_encodings))

2


In [97]:
tokens = tokenizer.tokenize("This statement shows bias.")
print(tokens)#BPE tokenization

['This', 'Ġstatement', 'Ġshows', 'Ġbias', '.']


BatchEncoding (train_encodings)
│
├── dictionary data
│     ├── input_ids
│     └── attention_mask
│
└── encodings list
      ├── Encoding object
      ├── Encoding object
      └── Encoding object

In [98]:
print(train_texts[0])
print(train_encodings["input_ids"][0])
print(train_encodings["attention_mask"][0])

So let s remember The White House chief is just a dude He s not God or even a god Let s put the p s into proper grammatical context If anyone deserves a consistently capitalized letter outside of God that is it s the citizens who elect the taxpayers who employ the patriots who defend 
[0, 2847, 905, 579, 2145, 20, 735, 446, 834, 16, 95, 10, 22633, 91, 579, 45, 1840, 50, 190, 10, 9069, 2780, 579, 342, 5, 181, 579, 88, 4692, 25187, 45816, 5377, 318, 1268, 8613, 10, 6566, 812, 1538, 1601, 751, 9, 1840, 14, 16, 24, 579, 5, 2286, 54, 10371, 5, 7660, 54, 12735, 5, 39343, 5992, 54, 4538, 1437, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 

In [99]:
train_encodings.encodings[0].offsets

[(0, 0),
 (0, 2),
 (3, 6),
 (7, 8),
 (9, 17),
 (18, 21),
 (22, 27),
 (28, 33),
 (34, 39),
 (40, 42),
 (43, 47),
 (48, 49),
 (50, 54),
 (55, 57),
 (58, 59),
 (60, 63),
 (64, 67),
 (68, 70),
 (71, 75),
 (76, 77),
 (78, 81),
 (82, 85),
 (86, 87),
 (88, 91),
 (92, 95),
 (96, 97),
 (98, 99),
 (100, 104),
 (105, 111),
 (112, 116),
 (116, 123),
 (124, 131),
 (132, 134),
 (135, 141),
 (142, 150),
 (151, 152),
 (153, 165),
 (166, 173),
 (173, 177),
 (178, 184),
 (185, 192),
 (193, 195),
 (196, 199),
 (200, 204),
 (205, 207),
 (208, 210),
 (211, 212),
 (213, 216),
 (217, 225),
 (226, 229),
 (230, 235),
 (236, 239),
 (240, 249),
 (250, 253),
 (254, 260),
 (261, 264),
 (265, 270),
 (270, 273),
 (274, 277),
 (278, 284),
 (285, 285),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),
 (0, 0),


In [100]:
train_encodings.encodings[0]

Encoding(num_tokens=128, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])

In [101]:
import torch
from torch.utils.data import Dataset

class BiasDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [102]:
train_dataset = BiasDataset(train_encodings, train_labels)
val_dataset = BiasDataset(val_encodings, val_labels)
test_dataset = BiasDataset(test_encodings, test_labels)

In [103]:
print("Train dataset size:", len(train_dataset))
print("Validation dataset size:", len(val_dataset))
print("Test dataset size:", len(test_dataset))

Train dataset size: 19602
Validation dataset size: 2178
Test dataset size: 5445


''''You are training a RoBERTa model for bias detection, which is a deep learning model. Deep learning models work with tensors (multi-dimensional arrays) and require automatic differentiation to update their parameters. That’s exactly what PyTorch provides. The Trainer API you are using internally expects PyTorch datasets and tensors''''

In [104]:
from transformers import RobertaForSequenceClassification

model = RobertaForSequenceClassification.from_pretrained(
"roberta-base",
num_labels=4
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [105]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    bias="none",
    target_modules=["query", "value"]
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 888,580 || all params: 125,537,288 || trainable%: 0.7078


In [ ]:
#eee
from transformers import TrainingArguments

training_args = TrainingArguments(
output_dir="./results",
num_train_epochs=3,
per_device_train_batch_size=16,
per_device_eval_batch_size=16,
eval_strategy="epoch",
learning_rate=2e-5,
logging_dir="./logs"
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [107]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="no",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
    load_best_model_at_end=False,
    report_to="none"
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [108]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import numpy as np

def compute_metrics(eval_pred):
    logits, labels = eval_pred

    # Convert raw logits to predicted class indices
    preds = np.argmax(logits, axis=1)

    # Calculate metrics
    acc = accuracy_score(labels, preds)
    precision = precision_score(labels, preds, average="weighted", zero_division=0)  # weighted handles class imbalance
    recall = recall_score(labels, preds, average="weighted", zero_division=0)
    f1 = f1_score(labels, preds, average="weighted", zero_division=0)

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }


'''''Two things happen:

1️⃣ The pretrained RoBERTa encoder weights are loaded 2️⃣ A new classification head is added on top

So the architecture becomes: Input sentence ↓ RoBERTa encoder (pretrained) ↓ Sentence embedding ↓ NEW classification head ↓ Bias class prediction


Instead, the model was trained using a self-supervised task called Masked Language Modeling.''''''


In [ ]:
#eeee
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average='weighted'
    )
    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

In [109]:
import time
from transformers import TrainerCallback

class EpochTimeCallback(TrainerCallback):
    def __init__(self):
        self.epoch_start_time = None

    def on_epoch_begin(self, args, state, control, **kwargs):
        self.epoch_start_time = time.time()

    def on_epoch_end(self, args, state, control, **kwargs):
        epoch_time = time.time() - self.epoch_start_time
        print(f"Epoch {int(state.epoch)} runtime: {epoch_time:.2f} seconds")

In [110]:
from transformers import Trainer

trainer = Trainer(
model=model,
args=training_args,
train_dataset=train_dataset,
eval_dataset=val_dataset,
compute_metrics=compute_metrics,
callbacks=[EpochTimeCallback()]
)


In [111]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.580145,0.532534,0.746556,0.635720,0.746556,0.678336
2,0.519110,0.507127,0.750230,0.723626,0.750230,0.722267
3,0.521320,0.500827,0.764922,0.754424,0.764922,0.709721


Epoch 1 runtime: 321.73 seconds
Epoch 2 runtime: 318.27 seconds
Epoch 3 runtime: 318.49 seconds


TrainOutput(global_step=7353, training_loss=0.5968479222381065, metrics={'train_runtime': 1005.8463, 'train_samples_per_second': 58.464, 'train_steps_per_second': 7.31, 'total_flos': 3908327586729984.0, 'train_loss': 0.5968479222381065, 'epoch': 3.0})

In [112]:
test_results = trainer.evaluate(test_dataset)
print(test_results)

{'eval_loss': 0.4857631325721741, 'eval_accuracy': 0.7608815426997245, 'eval_precision': 0.7280743069900635, 'eval_recall': 0.7608815426997245, 'eval_f1': 0.7042727833503716, 'eval_runtime': 35.7092, 'eval_samples_per_second': 152.482, 'eval_steps_per_second': 19.071, 'epoch': 3.0}


In [116]:
import torch

# 🔹 Set device (GPU if available)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# 🔹 Move model to device
model.to(device)
model.eval()

# 🔹 Sample texts
texts = [
    "Women are naturally bad at coding.",
    "All politicians are corrupt.",
    "All politicians are corrupt and women are naturally bad at coding."
]

# 🔹 Tokenize
inputs = tokenizer(
    texts,
    return_tensors="pt",
    truncation=True,
    padding=True,
    max_length=128
)

# 🔹 Move inputs to device
inputs = {k: v.to(device) for k, v in inputs.items()}

# 🔹 Forward pass
with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits

# 🔹 Convert to probabilities
probs = torch.softmax(logits, dim=-1)
preds = torch.argmax(probs, dim=1)

# 🔹 Label mapping
label_map = {
    0: "no_bias",
    1: "gender",
    2: "racial",
    3: "political"
}

# 🔹 Convert predictions to labels
pred_labels = [label_map[p.item()] for p in preds]

# 🔹 Print results
for text, label, prob in zip(texts, pred_labels, probs):
    print(f"\nText: {text}")
    print(f"Predicted bias: {label}")
    print("Probabilities:")
    for i, p in enumerate(prob):
        print(f"  {label_map[i]}: {p.item():.4f}")
    print("-" * 50)

Using device: cuda

Text: Women are naturally bad at coding.
Predicted bias: gender
Probabilities:
  no_bias: 0.0002
  gender: 0.9672
  racial: 0.0320
  political: 0.0007
--------------------------------------------------

Text: All politicians are corrupt.
Predicted bias: political
Probabilities:
  no_bias: 0.0393
  gender: 0.0113
  racial: 0.0065
  political: 0.9428
--------------------------------------------------

Text: All politicians are corrupt and women are naturally bad at coding.
Predicted bias: gender
Probabilities:
  no_bias: 0.0006
  gender: 0.9639
  racial: 0.0333
  political: 0.0022
--------------------------------------------------


In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [ ]:
model.to(device)  # model now lives on GPU (or CPU if no GPU)

PeftModelForSequenceClassification(
  (base_model): LoraModel(
    (model): RobertaForSequenceClassification(
      (roberta): RobertaModel(
        (embeddings): RobertaEmbeddings(
          (word_embeddings): Embedding(50265, 768, padding_idx=1)
          (token_type_embeddings): Embedding(1, 768)
          (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
          (position_embeddings): Embedding(514, 768, padding_idx=1)
        )
        (encoder): RobertaEncoder(
          (layer): ModuleList(
            (0-11): 12 x RobertaLayer(
              (attention): RobertaAttention(
                (self): RobertaSelfAttention(
                  (query): lora.Linear(
                    (base_layer): Linear(in_features=768, out_features=768, bias=True)
                    (lora_dropout): ModuleDict(
                      (default): Dropout(p=0.1, inplace=False)
                    )
                    (lora_A): ModuleD

In [ ]:
inputs = {k: v.to(device) for k, v in inputs.items()}

In [ ]:
texts = [
    "Women are naturally bad at coding.",
    "All politicians are corrupt.",
    "All politicians are corrupt. and Women are naturally bad at coding.",
]

In [ ]:
inputs = tokenizer(
    texts,
    return_tensors="pt",       # PyTorch tensors
    truncation=True,           # truncate if too long
    padding=True               # pad to same length
)

In [ ]:
import torch

outputs = model(**inputs)
logits = outputs.logits

In [ ]:
probs = torch.softmax(logits, dim=-1)      # convert to probabilities
preds = torch.argmax(probs, dim=1)         # predicted class indices

In [ ]:
label_map = {
    0: "no_bias",
    1: "gender",
    2: "racial",
    3: "political"
}

pred_labels = [label_map[p.item()] for p in preds]

In [ ]:
for text, label, prob in zip(texts, pred_labels, probs):
    print(f"\nText: {text}")
    print(f"Predicted bias: {label}")
    print("Probabilities:")
    for i, p in enumerate(prob):
        print(f"  {label_map[i]}: {p.item():.4f}")
    print("-" * 50)

Text: Women are naturally bad at coding.
Predicted bias: gender
Probabilities: [2.7208985557081178e-05, 0.989003598690033, 0.010564679279923439, 0.00040455436101183295]
--------------------------------------------------
Text: All politicians are corrupt.
Predicted bias: political
Probabilities: [0.012528983876109123, 0.005633845459669828, 0.001701868837699294, 0.9801352620124817]
--------------------------------------------------
Text: All politicians are corrupt. and Women are naturally bad at coding.
Predicted bias: gender
Probabilities: [9.602370846550912e-05, 0.9917601943016052, 0.007690142840147018, 0.0004535928019322455]
--------------------------------------------------


to save the pretrained model

to load the pretrained model


finetunning only classification head

In [37]:
from transformers import RobertaForSequenceClassification

model2 = RobertaForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=4
)

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Freeze ALL parameters

In [38]:
for param in model2.parameters():
    param.requires_grad = False

Unfreeze ONLY classification head

In [39]:
for param in model2.classifier.parameters():
    param.requires_grad = True

In [40]:
def print_trainable_parameters(model):
    trainable_params = 0
    all_params = 0

    for param in model2.parameters():
        num_params = param.numel()
        all_params += num_params
        if param.requires_grad:
            trainable_params += num_params

    print(
        f"trainable params: {trainable_params:,} || "
        f"all params: {all_params:,} || "
        f"trainable%: {100 * trainable_params / all_params:.4f}"
    )

In [41]:
print_trainable_parameters(model2)

trainable params: 593,668 || all params: 124,648,708 || trainable%: 0.4763


In [63]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import numpy as np

def compute_metrics(eval_pred):
    logits, labels = eval_pred

    # Convert raw logits to predicted class indices
    preds = np.argmax(logits, axis=1)

    # Calculate metrics
    acc = accuracy_score(labels, preds)
    precision = precision_score(labels, preds, average="weighted",zero_division=0)  # weighted handles class imbalance
    recall = recall_score(labels, preds, average="weighted",zero_division=0)
    f1 = f1_score(labels, preds, average="weighted",zero_division=0)

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [52]:
import time
from transformers import TrainerCallback

class EpochTimeCallback(TrainerCallback):
    def __init__(self):
        self.start_time = None

    def on_epoch_begin(self, args, state, control, **kwargs):
        self.start_time = time.time()

    def on_epoch_end(self, args, state, control, **kwargs):
        epoch_time = time.time() - self.start_time
        print(f"Epoch {int(state.epoch)} runtime: {epoch_time:.2f} seconds")

In [62]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="no",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
    load_best_model_at_end=False,
    report_to="none",
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [65]:
from transformers import Trainer

trainer = Trainer(
model=model2,
args=training_args,
train_dataset=train_dataset,
eval_dataset=val_dataset,
compute_metrics=compute_metrics,
callbacks=[EpochTimeCallback()]
)


In [66]:
trainer.train()


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.743142,0.667094,0.715335,0.604167,0.715335,0.647077
2,0.731015,0.659413,0.713039,0.600271,0.713039,0.643410
3,0.748351,0.656085,0.716253,0.601991,0.716253,0.647298


Epoch 1 runtime: 154.28 seconds
Epoch 2 runtime: 151.10 seconds
Epoch 3 runtime: 151.80 seconds


TrainOutput(global_step=7353, training_loss=0.7385043939668013, metrics={'train_runtime': 502.242, 'train_samples_per_second': 117.087, 'train_steps_per_second': 14.64, 'total_flos': 3868196641081344.0, 'train_loss': 0.7385043939668013, 'epoch': 3.0})

In [67]:
test_results = trainer.evaluate(test_dataset)
print(test_results)

{'eval_loss': 0.649217426776886, 'eval_accuracy': 0.7173553719008264, 'eval_precision': 0.6039007448028039, 'eval_recall': 0.7173553719008264, 'eval_f1': 0.6483120876375587, 'eval_runtime': 36.7655, 'eval_samples_per_second': 148.101, 'eval_steps_per_second': 18.523, 'epoch': 3.0}


In [71]:
import torch

# 🔹 Set device (GPU if available)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# 🔹 Move model to device
model2.to(device)
model2.eval()

# 🔹 Sample texts
texts = [
    "Women are naturally bad at driving.",
    "All politicians are corrupt.",
    "All politicians are corrupt and women are naturally bad at coding."
]

# 🔹 Tokenize
inputs = tokenizer(
    texts,
    return_tensors="pt",
    truncation=True,
    padding=True,
    max_length=128
)

# 🔹 Move inputs to device
inputs = {k: v.to(device) for k, v in inputs.items()}

# 🔹 Forward pass
with torch.no_grad():
    outputs = model2(**inputs)
    logits = outputs.logits

# 🔹 Convert to probabilities
probs = torch.softmax(logits, dim=-1)
preds = torch.argmax(probs, dim=1)

# 🔹 Label mapping
label_map = {
    0: "no_bias",
    1: "gender",
    2: "racial",
    3: "political"
}

# 🔹 Convert predictions to labels
pred_labels = [label_map[p.item()] for p in preds]

# 🔹 Print results
for text, label, prob in zip(texts, pred_labels, probs):
    print(f"\nText: {text}")
    print(f"Predicted bias: {label}")
    print("Probabilities:")
    for i, p in enumerate(prob):
        print(f"  {label_map[i]}: {p.item():.4f}")
    print("-" * 50)

Using device: cuda

Text: Women are naturally bad at driving.
Predicted bias: gender
Probabilities:
  no_bias: 0.0048
  gender: 0.7876
  racial: 0.1792
  political: 0.0284
--------------------------------------------------

Text: All politicians are corrupt.
Predicted bias: political
Probabilities:
  no_bias: 0.0330
  gender: 0.2596
  racial: 0.1106
  political: 0.5969
--------------------------------------------------

Text: All politicians are corrupt and women are naturally bad at coding.
Predicted bias: gender
Probabilities:
  no_bias: 0.0025
  gender: 0.7830
  racial: 0.2053
  political: 0.0092
--------------------------------------------------
